# 04 · Unsupervised anomaly detection — model comparison
IsolationForest, LOF, Autoencoder, and MiniLM-embeddings + IForest,
trained without labels and evaluated (PR-AUC / ROC-AUC / F1) against the
weak labels.


In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from src.config import SAMPLE_FILE
from src.data.parse import load_file
df = load_file(SAMPLE_FILE)
print(df.shape); df.head(3)


In [ ]:
from src.labeling.mitre_attack import label_events
from src.data.features import add_behavioral_features
from src.models.unsupervised import IsolationForestModel, LOFModel
from src.models.autoencoder import AutoencoderModel
from src.models.embed import EmbeddingIForestModel
from src.models.evaluate import evaluate_scores, comparison_table
df = add_behavioral_features(label_events(df)); y = df['label'].values


In [ ]:
models = [IsolationForestModel(), LOFModel(), AutoencoderModel(), EmbeddingIForestModel()]
results = {}
for m in models:
    m.fit(df); results[m.name] = evaluate_scores(y, m.score(df))
print(comparison_table(results))


## PR-AUC comparison


In [ ]:
import pandas as pd
pd.Series({k: v['pr_auc'] for k,v in results.items()}).sort_values().plot.barh(); plt.title('PR-AUC by model'); plt.show()


The best detector is persisted by `python -m src.pipeline` and served by
`src/serving/app.py` for streaming scoring of live CloudTrail events.
